In [1]:
import pandas as pd
import numpy as np
import os

os.makedirs('../data/output', exist_ok=True)

df_master = pd.read_parquet('../data/processed/master_table.parquet')
print(f"✅ df_master carregada: {df_master.shape}")

✅ df_master carregada: (99441, 34)


In [2]:
# ── DIM_CLIENTE ──────────────────────────────────────────────────────────
dim_cliente = df_master[[
    'customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
    'customer_city', 'customer_state'
]].drop_duplicates(subset=['customer_id']).copy()

# ── DIM_VENDEDOR ─────────────────────────────────────────────────────────
dim_vendedor = df_master[[
    'seller_id_principal', 'seller_city', 'seller_state'
]].drop_duplicates(subset=['seller_id_principal']).copy()
dim_vendedor = dim_vendedor.rename(columns={'seller_id_principal': 'seller_id'})

# ── DIM_PRODUTO ──────────────────────────────────────────────────────────
df_products = pd.read_csv('../data/raw/olist_products_dataset.csv')
df_translation = pd.read_csv('../data/raw/product_category_name_translation.csv')
dim_produto = pd.merge(df_products, df_translation, on='product_category_name', how='left')
dim_produto['product_category_name_english'] = (
    dim_produto['product_category_name_english']
    .fillna(dim_produto['product_category_name'])
    .fillna('nao_definido')
)
dim_produto = dim_produto[[
    'product_id', 'product_category_name_english',
    'product_weight_g', 'product_length_cm',
    'product_height_cm', 'product_width_cm'
]].copy()

# ── DIM_CALENDARIO ───────────────────────────────────────────────────────
data_range = pd.date_range(start='2016-01-01', end='2018-12-31', freq='D')
dim_calendario = pd.DataFrame({'data': data_range})
dim_calendario['ano'] = dim_calendario['data'].dt.year
dim_calendario['mes'] = dim_calendario['data'].dt.month
dim_calendario['dia'] = dim_calendario['data'].dt.day
dim_calendario['trimestre'] = dim_calendario['data'].dt.quarter
dim_calendario['semana_do_ano'] = dim_calendario['data'].dt.isocalendar().week.astype(int)
dim_calendario['dia_da_semana'] = dim_calendario['data'].dt.dayofweek
meses_pt = {1:'Janeiro', 2:'Fevereiro', 3:'Março', 4:'Abril', 5:'Maio',
            6:'Junho', 7:'Julho', 8:'Agosto', 9:'Setembro',
            10:'Outubro', 11:'Novembro', 12:'Dezembro'}
dim_calendario['nome_mes'] = dim_calendario['mes'].map(meses_pt)
dim_calendario['is_fim_semana'] = (dim_calendario['dia_da_semana'] >= 5).astype(int)
# Manter data como string no formato correto para Power BI
dim_calendario['data'] = dim_calendario['data'].dt.strftime('%Y-%m-%d')

# ── PRODUCT_ID NA MASTER ─────────────────────────────────────────────────
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
product_id_por_pedido = (
    order_items.groupby('order_id')['product_id']
    .first().reset_index()
)
if 'product_id' not in df_master.columns:
    df_master = df_master.merge(product_id_por_pedido, on='order_id', how='left')

# ── FATO_PEDIDOS ─────────────────────────────────────────────────────────
colunas_fato = [
    'order_id', 'customer_id', 'seller_id_principal', 'product_id',
    'order_status', 'order_purchase_timestamp',
    'qtd_itens', 'valor_total_produtos', 'valor_total_frete',
    'review_score', 'foi_entregue', 'pedido_atrasado',
    'dias_prazo_prometido', 'dias_entrega_real', 'dias_atraso', 'tipo_pagamento_principal', 'parcelas', 
]
colunas_presentes = [c for c in colunas_fato if c in df_master.columns]
fato_pedidos = df_master[colunas_presentes].copy()
fato_pedidos = fato_pedidos.rename(columns={'seller_id_principal': 'seller_id'})

# Criar chave de data como string para relacionamento com dim_calendario
fato_pedidos['order_purchase_timestamp'] = pd.to_datetime(fato_pedidos['order_purchase_timestamp'])
fato_pedidos['data_compra_fk'] = fato_pedidos['order_purchase_timestamp'].dt.strftime('%Y-%m-%d')

# Verificação crítica
assert len(fato_pedidos) == len(df_master), "ERRO: fato com linhas diferentes da master!"
print(f"✅ dim_cliente: {dim_cliente.shape}")
print(f"✅ dim_vendedor: {dim_vendedor.shape}")
print(f"✅ dim_produto: {dim_produto.shape}")
print(f"✅ dim_calendario: {dim_calendario.shape}")
print(f"✅ fato_pedidos: {fato_pedidos.shape}")
print(f"\nColunas fato: {fato_pedidos.columns.tolist()}")

✅ dim_cliente: (99441, 5)
✅ dim_vendedor: (3089, 3)
✅ dim_produto: (32951, 6)
✅ dim_calendario: (1096, 9)
✅ fato_pedidos: (99441, 18)

Colunas fato: ['order_id', 'customer_id', 'seller_id', 'product_id', 'order_status', 'order_purchase_timestamp', 'qtd_itens', 'valor_total_produtos', 'valor_total_frete', 'review_score', 'foi_entregue', 'pedido_atrasado', 'dias_prazo_prometido', 'dias_entrega_real', 'dias_atraso', 'tipo_pagamento_principal', 'parcelas', 'data_compra_fk']


In [3]:
OUTPUT = '../data/output/'

dim_cliente.to_csv(f'{OUTPUT}dim_cliente.csv', index=False, 
                   sep=';', decimal=',', encoding='utf-8-sig')
dim_vendedor.to_csv(f'{OUTPUT}dim_vendedor.csv', index=False, 
                    sep=';', decimal=',', encoding='utf-8-sig')
dim_produto.to_csv(f'{OUTPUT}dim_produto.csv', index=False, 
                   sep=';', decimal=',', encoding='utf-8-sig')
dim_calendario.to_csv(f'{OUTPUT}dim_calendario.csv', index=False, 
                      sep=';', decimal=',', encoding='utf-8-sig')
fato_pedidos.to_csv(f'{OUTPUT}fato_pedidos.csv', index=False, 
                    sep=';', decimal=',', encoding='utf-8-sig')

print("✅ Todos os CSVs exportados com separador ponto e vírgula!")
print("\nVerificação final:")
for tabela in ['fato_pedidos', 'dim_cliente', 'dim_vendedor', 'dim_produto', 'dim_calendario']:
    df = pd.read_csv(f'{OUTPUT}{tabela}.csv', sep=';', decimal=',')
    print(f"  {tabela}: {df.shape}")

✅ Todos os CSVs exportados com separador ponto e vírgula!

Verificação final:
  fato_pedidos: (99441, 18)
  dim_cliente: (99441, 5)
  dim_vendedor: (3089, 3)
  dim_produto: (32951, 6)
  dim_calendario: (1096, 9)
